# Lab 5: Supervisor + Deploy to AgentCore Runtime

The Supervisor orchestrates safety → retrieval → escalation → verification with step/time
budgets. We then deploy it to **AgentCore Runtime** using the `bedrock-agentcore-starter-toolkit`
`Runtime()` class — **entirely from Python**, no Node CLI.

The deployable entrypoint lives in `lab_helpers/runtime_entrypoint.py`.

### Step 1: Run the Supervisor locally first

In [ ]:
import importlib
import lab_helpers.runtime_entrypoint as rt
importlib.reload(rt)
import asyncio

async def ask(q):
    return await rt.invoke({"prompt": q})

print(asyncio.get_event_loop().run_until_complete(
    ask("What are the visiting hours at Riverside Health?")))
print("---")
print(asyncio.get_event_loop().run_until_complete(
    ask("I have a colonoscopy next Tuesday and I am almost out of my metformin. "
        "How do I prepare, can I get my refill in time, and should I stop taking it?")))

### Step 2: Deploy to AgentCore Runtime with the starter toolkit

This builds a container, pushes to ECR, and creates the managed runtime — all from Python.

In [ ]:
import boto3
from bedrock_agentcore_starter_toolkit import Runtime
import lab_helpers.utils as u

exec_role = u.create_agentcore_runtime_execution_role()
runtime = Runtime()

runtime.configure(
    entrypoint="lab_helpers/runtime_entrypoint.py",
    execution_role=exec_role,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=u.REGION,
    agent_name=u.RUNTIME_AGENT_NAME,
)
print("Configured runtime.")

In [ ]:
launch_result = runtime.launch()
print("Launched:", launch_result.agent_arn)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/runtime_arn", launch_result.agent_arn)

In [ ]:
import time
while True:
    st = runtime.status()
    ep = getattr(st, "endpoint", None)
    print("status:", ep.get("status") if ep else "provisioning...")
    if ep and ep.get("status") in ("READY", "ACTIVE"):
        break
    time.sleep(20)

### Step 3: Invoke the deployed runtime

In [ ]:
agentcore = boto3.client("bedrock-agentcore", region_name=u.REGION)
arn = u.get_ssm_parameter(f"{u.SSM_PREFIX}/runtime_arn")
resp = agentcore.invoke_agent_runtime(
    agentRuntimeArn=arn,
    payload=json.dumps({"prompt": "How should I prepare for my colonoscopy?"}).encode())
print(resp["response"].read().decode())

## Lab 5 complete ✅

Supervisor deployed to a managed AgentCore Runtime endpoint, invoked from the SDK.